# Vehicle Classification Data Collection

In [ ]:
import json
import shutil
from collections import Counter
from pathlib import Path


# ============================================================
# CẤU HÌNH - Sửa các đường dẫn dưới đây cho phù hợp
# ============================================================
CLASS_INFO_PATH = Path("/mnt/c/Users/Admin/Downloads/Car-1000-dataset/Car-1000-dataset/cls_info/class_info.json")
SOURCE_DIR      = Path("/mnt/c/Users/Admin/Downloads/Car-1000-dataset/Car-1000-dataset/all_images")   # <-- sửa đường dẫn này
OUTPUT_DIR      = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset")   # <-- sửa đường dẫn này
DRY_RUN         = False  # True = chỉ in kế hoạch, không copy file

# Các đuôi file ảnh sẽ copy
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".gif",
    ".webp", ".tif", ".tiff",
}


def load_id_to_main_type(class_info_path: Path) -> dict:
    """Đọc class_info.json, trả về dict {id: main_type}."""
    with open(class_info_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    id_to_main = {}
    for item in data:
        cls_id = item["id"]
        type_info = item.get("type_info", "")

        # "卡车####轻卡==Truck####Light Truck" -> "Truck####Light Truck" -> "Truck"
        en_part = type_info.split("==")[1] if "==" in type_info else type_info
        main_type = en_part.split("####")[0].strip() if "####" in en_part else en_part.strip()

        if not main_type:
            print(f"[WARN] id {cls_id} không có main_type hợp lệ, bỏ qua.")
            continue
        id_to_main[cls_id] = main_type
    return id_to_main


def copy_images(source_dir: Path, output_dir: Path, id_to_main: dict, dry_run: bool) -> None:
    """Duyệt source_dir, copy ảnh thẳng vào output_dir/<main_type>/ (không sub-folder id)."""
    stats = Counter()
    skipped_no_mapping = []
    empty_folders = []
    rename_conflicts = 0  # số lần phải thêm số đuôi do trùng tên

    id_folders = sorted(p for p in source_dir.iterdir() if p.is_dir())
    if not id_folders:
        print(f"[ERROR] Không tìm thấy sub-folder nào trong {source_dir}")
        return

    print(f"Tìm thấy {len(id_folders)} folder trong source.\n")

    for id_folder in id_folders:
        cls_id = id_folder.name

        main_type = id_to_main.get(cls_id)
        if main_type is None:
            skipped_no_mapping.append(cls_id)
            continue

        dest_folder = output_dir / main_type

        images = [
            p for p in id_folder.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]
        if not images:
            empty_folders.append(cls_id)
            continue

        if not dry_run:
            dest_folder.mkdir(parents=True, exist_ok=True)

        for img in images:
            # Đặt tên mới: <id>_<tên_gốc> để tránh trùng tên giữa các id khác nhau
            new_name = f"{cls_id}_{img.name}"
            dest_file = dest_folder / new_name

            # Nếu vẫn trùng (cùng id, tên file giống nhau do rglob bên trong), thêm hậu tố _1, _2...
            if dest_file.exists() and not dry_run:
                rename_conflicts += 1
                stem, suffix = dest_file.stem, dest_file.suffix
                counter = 1
                while dest_file.exists():
                    dest_file = dest_folder / f"{stem}_{counter}{suffix}"
                    counter += 1

            if not dry_run:
                shutil.copy2(img, dest_file)
            stats[main_type] += 1

    # Báo cáo
    print("=" * 60)
    print("BÁO CÁO")
    print("=" * 60)
    total = sum(stats.values())
    print(f"\nTổng ảnh đã copy: {total}\n")
    print("Chi tiết theo main type:")
    for t, c in stats.most_common():
        print(f"  {t:15s}: {c:>6d} ảnh")

    if rename_conflicts:
        print(f"\n[i] {rename_conflicts} file bị trùng tên → đã tự thêm hậu tố _1, _2...")

    if skipped_no_mapping:
        print(f"\n[!] Bỏ qua {len(skipped_no_mapping)} folder không có trong class_info.json:")
        print(f"    {skipped_no_mapping[:10]}{' ...' if len(skipped_no_mapping) > 10 else ''}")

    if empty_folders:
        print(f"\n[!] {len(empty_folders)} folder không chứa ảnh:")
        print(f"    {empty_folders[:10]}{' ...' if len(empty_folders) > 10 else ''}")

    if dry_run:
        print("\n[DRY RUN] Chưa copy file thật. Đặt DRY_RUN = False để chạy thật.")


def main():
    print(f"Class info : {CLASS_INFO_PATH}")
    print(f"Source     : {SOURCE_DIR}")
    print(f"Output     : {OUTPUT_DIR}")
    print(f"Mode       : {'DRY RUN' if DRY_RUN else 'COPY'}\n")

    if not CLASS_INFO_PATH.is_file():
        print(f"[ERROR] Không tìm thấy file: {CLASS_INFO_PATH}")
        return
    if not SOURCE_DIR.is_dir():
        print(f"[ERROR] Folder source không tồn tại: {SOURCE_DIR}")
        return

    id_to_main = load_id_to_main_type(CLASS_INFO_PATH)
    print(f"Đã load {len(id_to_main)} id từ class_info.json.\n")

    if not DRY_RUN:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    copy_images(SOURCE_DIR, OUTPUT_DIR, id_to_main, dry_run=DRY_RUN)


if __name__ == "__main__":
    main()


Class info : /mnt/c/Users/Admin/Downloads/Car-1000-dataset/Car-1000-dataset/cls_info/class_info.json
Source     : /mnt/c/Users/Admin/Downloads/Car-1000-dataset/Car-1000-dataset/all_images
Output     : /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset
Mode       : COPY

Đã load 1000 id từ class_info.json.

Tìm thấy 1000 folder trong source.

BÁO CÁO

Tổng ảnh đã copy: 140267

Chi tiết theo main type:
  SUV            :  61618 ảnh
  Sedan          :  44702 ảnh
  Truck          :  11803 ảnh
  MPV            :   8247 ảnh
  Sports Car     :   6247 ảnh
  Bus            :   3858 ảnh
  Van            :   3792 ảnh


In [ ]:
# Dataset Splitting for Training

In [ ]:
import random
import shutil
from collections import Counter
from pathlib import Path


# ============================================================
# CẤU HÌNH
# ============================================================
SOURCE_DIR        = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset")   # <-- sửa
OUTPUT_DIR        = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short-1500")         # <-- sửa

N_SAMPLES_PER_CLASS = 1500           # số ảnh lấy ra mỗi class
TRAIN_RATIO         = 0.7           # 70%
VAL_RATIO           = 0.3           # 20%
TEST_RATIO          = 0          # 10%  (3 số phải cộng = 1.0)

SEED              = 42              # để sample reproducible
IMAGE_EXTENSIONS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
MOVE_INSTEAD_COPY = False           # True = move (nhanh hơn, xóa gốc), False = copy
DRY_RUN           = False           # True = chỉ in kế hoạch, không ghi file
# ============================================================


def split_indices(n: int, train_r: float, val_r: float):
    """Chia n thành 3 phần theo tỉ lệ train/val/test."""
    n_train = round(n * train_r)
    n_val   = round(n * val_r)
    n_test  = n - n_train - n_val   # phần còn lại = test (tránh sai số làm tròn)
    return n_train, n_val, n_test


def main():
    # Validate ratios
    total_ratio = TRAIN_RATIO + VAL_RATIO + TEST_RATIO
    if abs(total_ratio - 1.0) > 1e-6:
        print(f"[ERROR] Tổng ratio phải = 1.0, hiện tại = {total_ratio}")
        return

    if not SOURCE_DIR.is_dir():
        print(f"[ERROR] Source không tồn tại: {SOURCE_DIR}")
        return

    class_folders = sorted(p for p in SOURCE_DIR.iterdir() if p.is_dir())
    if not class_folders:
        print(f"[ERROR] Không tìm thấy class folder trong {SOURCE_DIR}")
        return

    print(f"Source       : {SOURCE_DIR}")
    print(f"Output       : {OUTPUT_DIR}")
    print(f"Classes      : {len(class_folders)} -> {[p.name for p in class_folders]}")
    print(f"Samples/class: {N_SAMPLES_PER_CLASS}")
    print(f"Split        : train={TRAIN_RATIO} / val={VAL_RATIO} / test={TEST_RATIO}")

    n_train, n_val, n_test = split_indices(N_SAMPLES_PER_CLASS, TRAIN_RATIO, VAL_RATIO)
    print(f"  -> train={n_train}, val={n_val}, test={n_test} per class\n")

    op      = "MOVE" if MOVE_INSTEAD_COPY else "COPY"
    action  = shutil.move if MOVE_INSTEAD_COPY else shutil.copy2
    print(f"Mode         : {'DRY RUN' if DRY_RUN else op}\n")

    if not DRY_RUN:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    random.seed(SEED)
    stats = {"train": Counter(), "val": Counter(), "test": Counter()}
    not_enough = []

    for class_dir in class_folders:
        class_name = class_dir.name

        # Lấy tất cả ảnh trong class folder (không đệ quy sâu, chỉ file trực tiếp)
        all_images = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if len(all_images) < N_SAMPLES_PER_CLASS:
            print(f"[!] {class_name}: chỉ có {len(all_images)} ảnh "
                  f"(< {N_SAMPLES_PER_CLASS}) -> dùng hết")
            not_enough.append((class_name, len(all_images)))
            sampled = all_images.copy()
            random.shuffle(sampled)
        else:
            sampled = random.sample(all_images, N_SAMPLES_PER_CLASS)

        # Split: dùng index để deterministic
        # sampled đã được shuffle (hoặc random.sample trả về ngẫu nhiên)
        # nếu class không đủ ảnh, split theo tỉ lệ trên số ảnh thực tế
        actual_n = len(sampled)
        if actual_n < N_SAMPLES_PER_CLASS:
            a_train, a_val, a_test = split_indices(actual_n, TRAIN_RATIO, VAL_RATIO)
        else:
            a_train, a_val, a_test = n_train, n_val, n_test

        train_files = sampled[:a_train]
        val_files   = sampled[a_train:a_train + a_val]
        test_files  = sampled[a_train + a_val:a_train + a_val + a_test]

        # Copy sang output_dir/<split>/<class>/
        for split_name, files in [("train", train_files),
                                  ("val", val_files),
                                  ("test", test_files)]:
            dest_dir = OUTPUT_DIR / split_name / class_name
            if not DRY_RUN:
                dest_dir.mkdir(parents=True, exist_ok=True)

            for src in files:
                dest = dest_dir / src.name
                # Xử lý trùng tên (hiếm, vì file đã đc đặt tên <id>_<name> ở bước trước)
                if dest.exists() and not DRY_RUN:
                    stem, suf = dest.stem, dest.suffix
                    i = 1
                    while dest.exists():
                        dest = dest_dir / f"{stem}_{i}{suf}"
                        i += 1

                if not DRY_RUN:
                    action(src, dest)
                stats[split_name][class_name] += 1

        print(f"  {class_name:12s}: train={a_train:3d}  val={a_val:3d}  test={a_test:3d}  "
              f"(tổng nguồn: {len(all_images)})")

    # Báo cáo
    print("\n" + "=" * 60)
    print("KẾT QUẢ")
    print("=" * 60)
    for split_name in ["train", "val", "test"]:
        total = sum(stats[split_name].values())
        print(f"\n{split_name.upper()} ({total} ảnh):")
        for cls, n in sorted(stats[split_name].items()):
            print(f"  {cls:12s}: {n}")

    total_all = sum(sum(c.values()) for c in stats.values())
    print(f"\nTổng cộng: {total_all} ảnh")

    if not_enough:
        print(f"\n[!] {len(not_enough)} class không đủ {N_SAMPLES_PER_CLASS} ảnh:")
        for name, n in not_enough:
            print(f"    {name}: {n}")

    if DRY_RUN:
        print("\n[DRY RUN] Chưa ghi file. Đặt DRY_RUN = False để chạy thật.")
    else:
        print(f"\n[✓] Dataset sẵn sàng tại: {OUTPUT_DIR}")
        print(f"    Dùng cho YOLO-cls: yolo classify train data={OUTPUT_DIR} ...")


if __name__ == "__main__":
    main()

Source       : /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset
Output       : /mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short-1500
Classes      : 9 -> ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']
Samples/class: 1500
Split        : train=0.7 / val=0.3 / test=0
  -> train=1050, val=450, test=0 per class

Mode         : COPY

  Bus         : train=1050  val=450  test=  0  (tổng nguồn: 3858)
  MPV         : train=1050  val=450  test=  0  (tổng nguồn: 8247)
  SUV         : train=1050  val=450  test=  0  (tổng nguồn: 61618)
  Sedan       : train=1050  val=450  test=  0  (tổng nguồn: 44702)
  Sports Car  : train=1050  val=450  test=  0  (tổng nguồn: 6247)
  Truck       : train=1050  val=450  test=  0  (tổng nguồn: 11803)
  Van         : train=1050  val=450  test=  0  (tổng nguồn: 3792)
  bicycle     : train=1050  val=450  test=  0  (tổng nguồ

# YOLOv26-cls training

In [8]:
! yolo classify train data=/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short model=yolo26s-cls.pt epochs=30 imgsz=224

New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.12.0 torch-2.9.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

In [ ]:
"""
Pipeline: Video -> YOLO detection -> ROI filter -> YOLO classification -> output video

"""

import json
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO


# ============================================================
# CẤU HÌNH
# ============================================================
VIDEO_INPUT    = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4")    # <-- sửa
VIDEO_OUTPUT   = Path("./output_video_our_roi.mp4")
ROI_JSON       = Path("./roi.json")                  # <-- từ click_to_set_roi.py

DET_MODEL_PATH = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/train/weights/best.pt")                  # YOLO-det (pretrained hoặc custom)
CLS_MODEL_PATH = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/classify/train3/weights/best.pt")  # <-- YOLO-cls đã train

# Class mà YOLO-det coi là "xe" (COCO: car=2, motorcycle=3, bus=5, truck=7)
# Đặt None nếu YOLO-det của bạn chỉ có 1 class vehicle.
VEHICLE_CLASS_IDS = None

DET_CONF_THRESHOLD   = 0.4
ROI_COVERAGE_THRESHOLD = 0.2   # bbox phải có >50% diện tích trong ROI thì mới classify
SHOW_REALTIME        = False
DEVICE               = 0  # 0 = GPU đầu, "cpu" nếu không có GPU

# ============================================================
# ---- MODE ----
# True  : chỉ classify các bbox có >= ROI_COVERAGE_THRESHOLD diện tích trong ROI
# False : classify TẤT CẢ bbox trên cả frame (không dùng ROI)
USE_ROI        = True
ROI_JSON       = Path("./roi.json")  # chỉ cần nếu USE_ROI = True
 
# ============================================================
 
 
# Màu (BGR)
COLOR_ROI      = (0, 255, 255)      # Vàng: ROI polygon
COLOR_VEHICLE  = (255, 128, 0)      # Xanh dương sáng: vehicle ngoài ROI
COLOR_TEXT     = (255, 255, 255)
 
# Bảng màu cho từng class trong ROI (sau khi classify)
# Key: tên class khớp với cls_model.names
# Value: BGR tuple - chọn sao cho 9 class tách biệt rõ, không đụng COLOR_VEHICLE
CLASS_COLORS = {
    "SUV":        (0, 255, 0),       # xanh lá
    "Sedan":      (0, 165, 255),     # cam
    "Truck":      (0, 0, 255),       # đỏ
    "MPV":        (255, 0, 255),     # magenta
    "Sports Car": (0, 255, 255),     # vàng
    "Bus":        (255, 255, 0),     # cyan
    "Van":        (128, 0, 128),     # tím đậm
    "motorcycle": (50, 50, 180),     # đỏ nâu (tách khỏi xanh dương của Vehicle)
    "bicycle":    (180, 105, 255),   # hồng đậm (tách khỏi Sedan/Sports Car)
}
# Màu dự phòng cho class không có trong bảng
COLOR_DEFAULT_CLS = (200, 200, 200)
 
 
def load_roi(json_path: Path, video_shape: tuple) -> np.ndarray:
    """
    Load ROI polygon từ JSON.
    Cảnh báo nếu kích thước video không khớp với lúc set ROI.
    """
    if not json_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ROI JSON: {json_path}")
 
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
 
    points = np.array(data["points"], dtype=np.int32)
 
    # Kiểm tra size khớp
    h, w = video_shape[:2]
    saved_w = data.get("frame_width")
    saved_h = data.get("frame_height")
    if saved_w and saved_h and (saved_w != w or saved_h != h):
        print(f"[!] CẢNH BÁO: ROI được set trên frame {saved_w}x{saved_h}, "
              f"video hiện tại là {w}x{h}. Polygon có thể lệch.")
 
    print(f"Loaded ROI: {len(points)} điểm từ {json_path}")
    return points
 
 
def bbox_roi_coverage(bbox, roi_mask) -> float:
    """
    Trả về tỉ lệ diện tích bbox nằm trong ROI, trong khoảng [0.0, 1.0].
    0.0 = không dính ROI, 1.0 = bbox hoàn toàn trong ROI.
 
    roi_mask: np.uint8 array cùng kích thước frame,
              pixel = 255 trong ROI, = 0 ngoài ROI. Precompute 1 lần.
    """
    h, w = roi_mask.shape
    x1, y1, x2, y2 = map(int, bbox)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return 0.0
 
    bbox_area = (x2 - x1) * (y2 - y1)
    # Số pixel của bbox nằm trong ROI = số pixel = 255 trong vùng bbox của mask
    inside_area = int((roi_mask[y1:y2, x1:x2] > 0).sum())
    return inside_area / bbox_area
 
 
def classify_crop(cls_model, crop_bgr):
    """YOLO-cls trên 1 crop. Trả về (label, confidence)."""
    if crop_bgr.size == 0:
        return None, 0.0
    results = cls_model.predict(crop_bgr, verbose=False, device=DEVICE)
    r = results[0]
    probs = r.probs
    top1_idx = int(probs.top1)
    top1_conf = float(probs.top1conf.item())
    return cls_model.names[top1_idx], top1_conf
 
 
def draw_roi(frame, roi_polygon):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [roi_polygon], COLOR_ROI)
    cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, dst=frame)
    cv2.polylines(frame, [roi_polygon], isClosed=True, color=COLOR_ROI, thickness=2)
    # Label "ROI" ở điểm đầu
    cv2.putText(frame, "ROI", tuple(roi_polygon[0] + np.array([5, -8])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_ROI, 2)
 
 
def draw_bbox(frame, bbox, color, label=None):
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    if label:
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(frame, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_TEXT, 2)
 
 
def draw_legend(frame, class_names, show_out_roi=True):
    """
    Vẽ legend (chú thích màu) ở góc phải trên của frame.
    show_out_roi: True nếu có dùng ROI (để hiện mục Vehicle out ROI), False nếu không.
    """
    h, w = frame.shape[:2]
    items = []
    if show_out_roi:
        items.append(("Vehicle (out ROI)", COLOR_VEHICLE))
    for name in class_names:
        items.append((name, CLASS_COLORS.get(name, COLOR_DEFAULT_CLS)))
 
    # Kích thước legend box
    line_h = 22
    pad = 8
    box_w = 220
    box_h = pad * 2 + line_h * len(items)
    x0 = w - box_w - 10
    y0 = 10
 
    # Nền bán trong suốt
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, dst=frame)
    cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (200, 200, 200), 1)
 
    # Từng dòng: ô màu + tên class
    for i, (name, color) in enumerate(items):
        y = y0 + pad + i * line_h + line_h // 2
        # Ô màu
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7), color, -1)
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7), (255, 255, 255), 1)
        # Tên
        cv2.putText(frame, name, (x0 + pad + 30, y + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_TEXT, 1, cv2.LINE_AA)
 
 
def main():
    # Validate
    required = [(VIDEO_INPUT, "video"),
                (DET_MODEL_PATH, "YOLO-det"),
                (CLS_MODEL_PATH, "YOLO-cls")]
    if USE_ROI:
        required.append((ROI_JSON, "ROI JSON"))
 
    for p, name in required:
        if not p.exists():
            print(f"[ERROR] Không tìm thấy {name}: {p}")
            return
 
    mode_str = f"USE_ROI=True (threshold={ROI_COVERAGE_THRESHOLD:.0%})" if USE_ROI \
               else "USE_ROI=False (xử lý cả frame)"
    print(f"Mode: {mode_str}\n")
 
    # Load models
    print("Đang load models...")
    det_model = YOLO(str(DET_MODEL_PATH))
    cls_model = YOLO(str(CLS_MODEL_PATH))
    print(f"  Det classes: {det_model.names}")
    print(f"  Cls classes: {cls_model.names}")
 
    # Mở video
    cap = cv2.VideoCapture(str(VIDEO_INPUT))
    if not cap.isOpened():
        print(f"[ERROR] Không mở được video: {VIDEO_INPUT}")
        return
 
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"\nVideo: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames")
 
    # Load ROI (chỉ khi USE_ROI)
    roi_polygon = None
    roi_mask = None
    if USE_ROI:
        roi_polygon = load_roi(ROI_JSON, (height, width))
        # Tạo ROI mask 1 lần dùng cho cả video (polygon không đổi)
        roi_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(roi_mask, [roi_polygon], 255)
 
    # Writer
    VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(VIDEO_OUTPUT), fourcc, fps, (width, height))
 
    frame_idx = 0
    stats = {"total_det": 0, "in_roi": 0, "classified": 0}
 
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
 
            # 1. YOLO-det
            det_kwargs = dict(conf=DET_CONF_THRESHOLD, verbose=False, device=DEVICE)
            if VEHICLE_CLASS_IDS is not None:
                det_kwargs["classes"] = VEHICLE_CLASS_IDS
            det_results = det_model.predict(frame, **det_kwargs)[0]
 
            # Vẽ ROI nếu có
            if USE_ROI:
                draw_roi(frame, roi_polygon)
 
            # 2+3. Duyệt bbox
            if det_results.boxes is not None and len(det_results.boxes) > 0:
                boxes = det_results.boxes.xyxy.cpu().numpy()
                confs = det_results.boxes.conf.cpu().numpy()
                clses = det_results.boxes.cls.cpu().numpy().astype(int)
 
                for bbox, det_conf, det_cls in zip(boxes, confs, clses):
                    stats["total_det"] += 1
                    x1, y1, x2, y2 = map(int, bbox)
 
                    if x2 <= x1 or y2 <= y1:
                        continue
 
                    # Filter ROI (chỉ nếu USE_ROI)
                    if USE_ROI:
                        coverage = bbox_roi_coverage((x1, y1, x2, y2), roi_mask)
                        if coverage < ROI_COVERAGE_THRESHOLD:
                            # Không đủ diện tích trong ROI: màu xanh vehicle, không classify
                            draw_bbox(frame, (x1, y1, x2, y2), COLOR_VEHICLE)
                            continue
 
                    stats["in_roi"] += 1   # "in_roi" = "được classify" (kể cả khi không dùng ROI)
 
                    # 4. Crop + classify
                    x1c, y1c = max(0, x1), max(0, y1)
                    x2c, y2c = min(width, x2), min(height, y2)
                    crop = frame[y1c:y2c, x1c:x2c]
 
                    cls_label, cls_conf = classify_crop(cls_model, crop)
                    if cls_label is not None:
                        stats["classified"] += 1
                        label_text = f"{cls_label} {cls_conf:.2f}"
                        box_color = CLASS_COLORS.get(cls_label, COLOR_DEFAULT_CLS)
                    else:
                        label_text = "?"
                        box_color = COLOR_DEFAULT_CLS
 
                    draw_bbox(frame, (x1, y1, x2, y2), box_color, label=label_text)
 
            # Overlay info
            info_label = "classified" if not USE_ROI else "in_roi"
            info = f"Frame {frame_idx}/{total_frames} | {info_label}: {stats['in_roi']}"
            cv2.putText(frame, info, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
 
            # Legend màu ở góc phải trên
            draw_legend(frame, list(cls_model.names.values()), show_out_roi=USE_ROI)
 
            writer.write(frame)
 
            if SHOW_REALTIME:
                cv2.imshow("Pipeline", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    print("\nDừng bởi người dùng.")
                    break
 
            if frame_idx % 30 == 0:
                pct = 100 * frame_idx / max(total_frames, 1)
                print(f"  Frame {frame_idx}/{total_frames} ({pct:.1f}%)  "
                      f"in_roi_sofar={stats['in_roi']}")
 
    finally:
        cap.release()
        writer.release()
        if SHOW_REALTIME:
            cv2.destroyAllWindows()
 
    # Báo cáo
    print("\n" + "=" * 60)
    print("HOÀN THÀNH")
    print("=" * 60)
    print(f"Frames xử lý        : {frame_idx}")
    print(f"Tổng detections     : {stats['total_det']}")
    if USE_ROI:
        print(f"  Pass ROI filter   : {stats['in_roi']}")
    print(f"  Đã classify       : {stats['classified']}")
    print(f"Video output        : {VIDEO_OUTPUT}")
 
 
if __name__ == "__main__":
    main() 


Mode: USE_ROI=True (threshold=20%)

Đang load models...
  Det classes: {0: 'vehicle'}
  Cls classes: {0: 'Bus', 1: 'MPV', 2: 'SUV', 3: 'Sedan', 4: 'Sports Car', 5: 'Truck', 6: 'Van', 7: 'bicycle', 8: 'motorcycle'}

Video: 1280x720 @ 30.0 FPS, 6023 frames
Loaded ROI: 4 điểm từ roi.json
  Frame 30/6023 (0.5%)  in_roi_sofar=0
  Frame 60/6023 (1.0%)  in_roi_sofar=36
  Frame 90/6023 (1.5%)  in_roi_sofar=116
  Frame 120/6023 (2.0%)  in_roi_sofar=161
  Frame 150/6023 (2.5%)  in_roi_sofar=216
  Frame 180/6023 (3.0%)  in_roi_sofar=274
  Frame 210/6023 (3.5%)  in_roi_sofar=335
  Frame 240/6023 (4.0%)  in_roi_sofar=376
  Frame 270/6023 (4.5%)  in_roi_sofar=392
  Frame 300/6023 (5.0%)  in_roi_sofar=448
  Frame 330/6023 (5.5%)  in_roi_sofar=465
  Frame 360/6023 (6.0%)  in_roi_sofar=504
  Frame 390/6023 (6.5%)  in_roi_sofar=571
  Frame 420/6023 (7.0%)  in_roi_sofar=603
  Frame 450/6023 (7.5%)  in_roi_sofar=648
  Frame 480/6023 (8.0%)  in_roi_sofar=749
  Frame 510/6023 (8.5%)  in_roi_sofar=844
  Fram

In [2]:
! yolo classify train data=/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short model=yolov12s-cls.pt epochs=30 imgsz=224

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12282MiB)
engine/trainer: task=classify, mode=train, model=yolov12s-cls.pt, data=/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short, epochs=30, time=None, patience=100, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train7, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=Fal

In [7]:
"""
Pipeline: Video -> YOLO detection -> ROI filter -> TIPSv2 classification -> output video

Thay đổi so với bản YOLO-cls:
    - Dùng google/tipsv2-b14 (zero-shot vision-language model) để classify crops
    - Không cần train classifier, chỉ cần định nghĩa CLASS_PROMPTS (text prompts)
    - Text embeddings precompute 1 lần, mỗi crop chỉ cần encode_image + matmul

Cài đặt:
    pip install ultralytics opencv-python numpy torch torchvision \
                transformers sentencepiece --break-system-packages

Workflow:
    Bước 1: (Optional) Chạy click_to_set_roi.py để tạo roi.json
    Bước 2: Chạy script này
"""

import json
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from transformers import AutoModel
from ultralytics import YOLO


# ============================================================
# CẤU HÌNH
# ============================================================
VIDEO_INPUT    = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4")
VIDEO_OUTPUT   = Path("./output_video_tipsv2.mp4")

DET_MODEL_PATH = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/train/weights/best.pt")
TIPS_MODEL_ID  = "google/tipsv2-b14"   # "google/tipsv2-b14" (base) | "google/tipsv2-l14" | "google/tipsv2-so400m-14" | "google/tipsv2-g14"
TIPS_IMG_SIZE  = 448                    # TIPSv2 dùng 448 (patch 14 -> 32×32 patches)

VEHICLE_CLASS_IDS = None
DET_CONF_THRESHOLD = 0.4

# ---- MODE ----
USE_ROI  = True
ROI_JSON = Path("./roi.json")
ROI_COVERAGE_THRESHOLD = 0.2

SHOW_REALTIME = True
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# ---- CLASS PROMPTS ----
# Mỗi class có thể có NHIỀU prompt, score sẽ lấy max/mean.
# Tune prompts này để cải thiện accuracy. Prompt càng cụ thể càng tốt.
CLASS_PROMPTS = {
    "SUV": [
        "a photo of an SUV",
        "a photo of a sport utility vehicle",
        "a photo of a large family car with high ground clearance",
    ],
    "Sedan": [
        "a photo of a sedan car",
        "a photo of a four-door passenger car",
    ],
    "Truck": [
        "a photo of a truck",
        "a photo of a freight truck",
        "a photo of a pickup truck",
    ],
    "MPV": [
        "a photo of an MPV",
        "a photo of a multi-purpose vehicle",
        "a photo of a minivan",
    ],
    "Sports Car": [
        "a photo of a sports car",
        "a photo of a low racing car",
    ],
    "Bus": [
        "a photo of a bus",
        "a photo of a city bus",
    ],
    "Van": [
        "a photo of a van",
        "a photo of a cargo van",
    ],
    "motorcycle": [
        "a photo of a motorcycle",
        "a photo of a motorbike",
    ],
    "bicycle": [
        "a photo of a bicycle",
        "a photo of a bike with pedals",
    ],
}
PROMPT_AGGREGATE = "mean"   # "mean" | "max" - cách tổng hợp nhiều prompt/class
CLS_CONF_THRESHOLD = 0.0    # chỉ hiển thị class nếu top1 similarity > ngưỡng này
# ============================================================


# Màu (BGR)
COLOR_ROI      = (0, 255, 255)
COLOR_VEHICLE  = (255, 128, 0)
COLOR_TEXT     = (255, 255, 255)

CLASS_COLORS = {
    "SUV":        (0, 255, 0),
    "Sedan":      (0, 165, 255),
    "Truck":      (0, 0, 255),
    "MPV":        (255, 0, 255),
    "Sports Car": (0, 255, 255),
    "Bus":        (255, 255, 0),
    "Van":        (128, 0, 128),
    "motorcycle": (50, 50, 180),
    "bicycle":    (180, 105, 255),
}
COLOR_DEFAULT_CLS = (200, 200, 200)


# ------------------------------------------------------------
# TIPSv2 classifier
# ------------------------------------------------------------
class TIPSv2Classifier:
    """
    Wrapper zero-shot classifier dùng TIPSv2.
    Precompute text embeddings 1 lần để tăng tốc.
    """

    def __init__(self, model_id, class_prompts, device="cuda",
                 img_size=448, aggregate="mean"):
        self.device = device
        self.aggregate = aggregate
        self.class_names = list(class_prompts.keys())

        print(f"Load {model_id}...")
        self.model = AutoModel.from_pretrained(
            model_id, trust_remote_code=True
        ).to(device).eval()

        # Image transform (TIPSv2: chỉ resize + ToTensor, KHÔNG normalize ImageNet)
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),  # -> [0, 1]
        ])

        # Precompute text embeddings: dict class_name -> tensor [num_prompts, embed_dim]
        self.text_embs = {}
        with torch.no_grad():
            for cls_name, prompts in class_prompts.items():
                emb = self.model.encode_text(prompts)  # [P, D]
                emb = F.normalize(emb, dim=-1)
                self.text_embs[cls_name] = emb.to(device)

        print(f"  Classes ({len(self.class_names)}): {self.class_names}")
        print(f"  Total prompts: {sum(len(p) for p in class_prompts.values())}")

    @torch.no_grad()
    def classify(self, crop_bgr):
        """
        Classify 1 crop (BGR numpy array).
        Trả về (label, confidence, probs_vector_numpy).
        """
        if crop_bgr is None or crop_bgr.size == 0:
            return None, 0.0, None

        # BGR -> RGB -> PIL -> tensor
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb)
        x = self.transform(pil).unsqueeze(0).to(self.device)

        # Encode image
        out = self.model.encode_image(x)
        img_emb = F.normalize(out.cls_token[:, 0, :], dim=-1)  # [1, D]

        # Similarity với text emb của từng class
        scores = []
        for cls_name in self.class_names:
            text_emb = self.text_embs[cls_name]                 # [P, D]
            sim = (img_emb @ text_emb.T).squeeze(0)             # [P]
            if self.aggregate == "mean":
                score = sim.mean().item()
            else:
                score = sim.max().item()
            scores.append(score)

        scores = np.array(scores, dtype=np.float32)
        # Softmax cho giống probability (chỉ để hiển thị, không ảnh hưởng argmax)
        probs = np.exp(scores) / np.exp(scores).sum()
        top_idx = int(probs.argmax())
        return self.class_names[top_idx], float(probs[top_idx]), probs


# ------------------------------------------------------------
# ROI helpers (giữ nguyên từ bản cũ)
# ------------------------------------------------------------
def load_roi(json_path, video_shape):
    if not json_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ROI JSON: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    points = np.array(data["points"], dtype=np.int32)
    h, w = video_shape[:2]
    sw, sh = data.get("frame_width"), data.get("frame_height")
    if sw and sh and (sw != w or sh != h):
        print(f"[!] ROI set trên {sw}x{sh}, video {w}x{h} - có thể lệch")
    print(f"Loaded ROI: {len(points)} điểm")
    return points


def bbox_roi_coverage(bbox, roi_mask):
    h, w = roi_mask.shape
    x1, y1, x2, y2 = map(int, bbox)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    bbox_area = (x2 - x1) * (y2 - y1)
    inside = int((roi_mask[y1:y2, x1:x2] > 0).sum())
    return inside / bbox_area


# ------------------------------------------------------------
# Drawing
# ------------------------------------------------------------
def draw_roi(frame, roi_polygon):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [roi_polygon], COLOR_ROI)
    cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, dst=frame)
    cv2.polylines(frame, [roi_polygon], True, COLOR_ROI, 2)
    cv2.putText(frame, "ROI", tuple(roi_polygon[0] + np.array([5, -8])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_ROI, 2)


def draw_bbox(frame, bbox, color, label=None):
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    if label:
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(frame, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_TEXT, 2)


def draw_legend(frame, class_names, show_out_roi=True):
    h, w = frame.shape[:2]
    items = []
    if show_out_roi:
        items.append(("Vehicle (out ROI)", COLOR_VEHICLE))
    for name in class_names:
        items.append((name, CLASS_COLORS.get(name, COLOR_DEFAULT_CLS)))

    line_h, pad, box_w = 22, 8, 220
    box_h = pad * 2 + line_h * len(items)
    x0, y0 = w - box_w - 10, 10

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, dst=frame)
    cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (200, 200, 200), 1)

    for i, (name, color) in enumerate(items):
        y = y0 + pad + i * line_h + line_h // 2
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7), color, -1)
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7),
                      (255, 255, 255), 1)
        cv2.putText(frame, name, (x0 + pad + 30, y + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_TEXT, 1, cv2.LINE_AA)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main():
    # Validate
    required = [(VIDEO_INPUT, "video"), (DET_MODEL_PATH, "YOLO-det")]
    if USE_ROI:
        required.append((ROI_JSON, "ROI JSON"))
    for p, name in required:
        if not p.exists():
            print(f"[ERROR] Không tìm thấy {name}: {p}")
            return

    mode_str = (f"USE_ROI=True (threshold={ROI_COVERAGE_THRESHOLD:.0%})"
                if USE_ROI else "USE_ROI=False (xử lý cả frame)")
    print(f"Mode: {mode_str}")
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # Load YOLO-det
    print("\nLoad YOLO-det...")
    det_model = YOLO(str(DET_MODEL_PATH))
    print(f"  Det classes: {det_model.names}")

    # Load TIPSv2
    print()
    cls_model = TIPSv2Classifier(
        TIPS_MODEL_ID, CLASS_PROMPTS,
        device=DEVICE, img_size=TIPS_IMG_SIZE, aggregate=PROMPT_AGGREGATE
    )

    # Mở video
    cap = cv2.VideoCapture(str(VIDEO_INPUT))
    if not cap.isOpened():
        print(f"[ERROR] Không mở được video: {VIDEO_INPUT}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"\nVideo: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames")

    # ROI
    roi_polygon, roi_mask = None, None
    if USE_ROI:
        roi_polygon = load_roi(ROI_JSON, (height, width))
        roi_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(roi_mask, [roi_polygon], 255)

    # Writer
    VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(VIDEO_OUTPUT),
                             cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (width, height))

    frame_idx = 0
    stats = {"total_det": 0, "in_roi": 0, "classified": 0}

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            # YOLO-det
            det_kwargs = dict(conf=DET_CONF_THRESHOLD, verbose=False, device=DEVICE)
            if VEHICLE_CLASS_IDS is not None:
                det_kwargs["classes"] = VEHICLE_CLASS_IDS
            det_results = det_model.predict(frame, **det_kwargs)[0]

            if USE_ROI:
                draw_roi(frame, roi_polygon)

            # Duyệt bbox
            if det_results.boxes is not None and len(det_results.boxes) > 0:
                boxes = det_results.boxes.xyxy.cpu().numpy()

                for bbox in boxes:
                    stats["total_det"] += 1
                    x1, y1, x2, y2 = map(int, bbox)
                    if x2 <= x1 or y2 <= y1:
                        continue

                    if USE_ROI:
                        coverage = bbox_roi_coverage((x1, y1, x2, y2), roi_mask)
                        if coverage < ROI_COVERAGE_THRESHOLD:
                            draw_bbox(frame, (x1, y1, x2, y2), COLOR_VEHICLE)
                            continue

                    stats["in_roi"] += 1

                    # Crop + TIPSv2 classify
                    x1c, y1c = max(0, x1), max(0, y1)
                    x2c, y2c = min(width, x2), min(height, y2)
                    crop = frame[y1c:y2c, x1c:x2c]

                    cls_label, cls_conf, _ = cls_model.classify(crop)

                    if cls_label is not None and cls_conf >= CLS_CONF_THRESHOLD:
                        stats["classified"] += 1
                        label_text = f"{cls_label} {cls_conf:.2f}"
                        box_color = CLASS_COLORS.get(cls_label, COLOR_DEFAULT_CLS)
                    else:
                        label_text = "?"
                        box_color = COLOR_DEFAULT_CLS

                    draw_bbox(frame, (x1, y1, x2, y2), box_color, label=label_text)

            # Overlay
            info_label = "classified" if not USE_ROI else "in_roi"
            info = f"Frame {frame_idx}/{total_frames} | {info_label}: {stats['in_roi']}"
            cv2.putText(frame, info, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            draw_legend(frame, cls_model.class_names, show_out_roi=USE_ROI)

            writer.write(frame)

            if SHOW_REALTIME:
                cv2.imshow("Pipeline (TIPSv2)", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    print("\nDừng bởi người dùng.")
                    break

            if frame_idx % 30 == 0:
                pct = 100 * frame_idx / max(total_frames, 1)
                print(f"  Frame {frame_idx}/{total_frames} ({pct:.1f}%)  "
                      f"classified_sofar={stats['classified']}")

    finally:
        cap.release()
        writer.release()
        if SHOW_REALTIME:
            cv2.destroyAllWindows()

    # Báo cáo
    print("\n" + "=" * 60)
    print("HOÀN THÀNH")
    print("=" * 60)
    print(f"Frames xử lý        : {frame_idx}")
    print(f"Tổng detections     : {stats['total_det']}")
    if USE_ROI:
        print(f"  Pass ROI filter   : {stats['in_roi']}")
    print(f"  Đã classify       : {stats['classified']}")
    print(f"Video output        : {VIDEO_OUTPUT}")


if __name__ == "__main__":
    main()

Mode: USE_ROI=True (threshold=20%)
Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti

Load YOLO-det...
  Det classes: {0: 'vehicle'}

Load google/tipsv2-b14...


Loading weights: 100%|██████████| 323/323 [00:00<00:00, 2502.46it/s]


  Classes (9): ['SUV', 'Sedan', 'Truck', 'MPV', 'Sports Car', 'Bus', 'Van', 'motorcycle', 'bicycle']
  Total prompts: 21

Video: 1280x720 @ 30.0 FPS, 6023 frames
Loaded ROI: 4 điểm
  Frame 30/6023 (0.5%)  classified_sofar=0
  Frame 60/6023 (1.0%)  classified_sofar=36
  Frame 90/6023 (1.5%)  classified_sofar=114
  Frame 120/6023 (2.0%)  classified_sofar=159
  Frame 150/6023 (2.5%)  classified_sofar=214
  Frame 180/6023 (3.0%)  classified_sofar=270
  Frame 210/6023 (3.5%)  classified_sofar=331
  Frame 240/6023 (4.0%)  classified_sofar=369
  Frame 270/6023 (4.5%)  classified_sofar=385
  Frame 300/6023 (5.0%)  classified_sofar=437
  Frame 330/6023 (5.5%)  classified_sofar=454
  Frame 360/6023 (6.0%)  classified_sofar=489
  Frame 390/6023 (6.5%)  classified_sofar=555
  Frame 420/6023 (7.0%)  classified_sofar=585
  Frame 450/6023 (7.5%)  classified_sofar=627
  Frame 480/6023 (8.0%)  classified_sofar=725
  Frame 510/6023 (8.5%)  classified_sofar=820
  Frame 540/6023 (9.0%)  classified_sofar=9

In [ ]:
"""
Pipeline: Video -> YOLO detection -> ROI filter -> CLIP fine-tuned classifier -> output video
"""

import json
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import CLIPImageProcessor, CLIPVisionModel
from ultralytics import YOLO


# ============================================================
# CẤU HÌNH
# ============================================================
VIDEO_INPUT    = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/2026-04-15 11-00-45.mp4")
VIDEO_OUTPUT   = Path("./output_video_clip_1500_full.mp4")

DET_MODEL_PATH = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/runs/detect/train/weights/best.pt")
CLIP_CHECKPOINT = Path("./clip_finetuned_1500/best_model.pt")   # <-- checkpoint từ train_clip_classifier.py

VEHICLE_CLASS_IDS = None
DET_CONF_THRESHOLD = 0.4

# ---- MODE ----
USE_ROI  = False
ROI_JSON = Path("./roi.json")
ROI_COVERAGE_THRESHOLD = 0.2

SHOW_REALTIME = False
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

CLS_CONF_THRESHOLD = 0.3    # chỉ hiển thị class nếu top1 prob > ngưỡng này
# ============================================================


# Màu (BGR)
COLOR_ROI      = (0, 255, 255)
COLOR_VEHICLE  = (255, 128, 0)
COLOR_TEXT     = (255, 255, 255)

CLASS_COLORS = {
    "SUV":        (0, 255, 0),
    "Sedan":      (0, 165, 255),
    "Truck":      (0, 0, 255),
    "MPV":        (255, 0, 255),
    "Sports Car": (0, 255, 255),
    "Bus":        (255, 255, 0),
    "Van":        (128, 0, 128),
    "motorcycle": (50, 50, 180),
    "bicycle":    (180, 105, 255),
}
COLOR_DEFAULT_CLS = (200, 200, 200)


# ------------------------------------------------------------
# CLIP Classifier (cùng kiến trúc với train_clip_classifier.py)
# ------------------------------------------------------------
class CLIPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained(model_name)
        hidden = self.vision.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, pixel_values):
        feats = self.vision(pixel_values=pixel_values).pooler_output
        return self.head(self.dropout(feats))


class CLIPInference:
    """
    Wrapper cho CLIP fine-tuned classifier.
    Load 1 lần, classify nhiều crop.
    """

    def __init__(self, checkpoint_path, device="cuda"):
        self.device = device

        print(f"Load CLIP checkpoint: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=device)
        self.class_names = ckpt["class_names"]
        self.model_name  = ckpt["model_name"]

        # Load model
        self.model = CLIPClassifier(
            self.model_name, len(self.class_names)
        ).to(device).eval()
        self.model.load_state_dict(ckpt["model_state"])

        # Transform giống như lúc train (resize + ToTensor + normalize CLIP)
        processor = CLIPImageProcessor.from_pretrained(self.model_name)
        size = processor.size.get("shortest_edge", 224)
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(processor.image_mean, processor.image_std),
        ])

        print(f"  Model: {self.model_name}")
        print(f"  Classes ({len(self.class_names)}): {self.class_names}")
        print(f"  Input size: {size}x{size}")

    @torch.no_grad()
    def classify(self, crop_bgr):
        """
        Classify 1 crop (BGR numpy array).
        Trả về (label, confidence, probs_vector_numpy).
        """
        if crop_bgr is None or crop_bgr.size == 0:
            return None, 0.0, None

        # BGR -> RGB -> PIL -> tensor (tương thích với pipeline train)
        rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(rgb)
        x = self.transform(pil).unsqueeze(0).to(self.device)

        logits = self.model(x)
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        top_idx = int(probs.argmax())
        return self.class_names[top_idx], float(probs[top_idx]), probs


# ------------------------------------------------------------
# ROI helpers
# ------------------------------------------------------------
def load_roi(json_path, video_shape):
    if not json_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ROI JSON: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    points = np.array(data["points"], dtype=np.int32)
    h, w = video_shape[:2]
    sw, sh = data.get("frame_width"), data.get("frame_height")
    if sw and sh and (sw != w or sh != h):
        print(f"[!] ROI set trên {sw}x{sh}, video {w}x{h} - có thể lệch")
    print(f"Loaded ROI: {len(points)} điểm")
    return points


def bbox_roi_coverage(bbox, roi_mask):
    h, w = roi_mask.shape
    x1, y1, x2, y2 = map(int, bbox)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    bbox_area = (x2 - x1) * (y2 - y1)
    inside = int((roi_mask[y1:y2, x1:x2] > 0).sum())
    return inside / bbox_area


# ------------------------------------------------------------
# Drawing
# ------------------------------------------------------------
def draw_roi(frame, roi_polygon):
    overlay = frame.copy()
    cv2.fillPoly(overlay, [roi_polygon], COLOR_ROI)
    cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, dst=frame)
    cv2.polylines(frame, [roi_polygon], True, COLOR_ROI, 2)
    cv2.putText(frame, "ROI", tuple(roi_polygon[0] + np.array([5, -8])),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_ROI, 2)


def draw_bbox(frame, bbox, color, label=None):
    x1, y1, x2, y2 = map(int, bbox)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    if label:
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(frame, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_TEXT, 2)


def draw_legend(frame, class_names, show_out_roi=True):
    h, w = frame.shape[:2]
    items = []
    if show_out_roi:
        items.append(("Vehicle (out ROI)", COLOR_VEHICLE))
    for name in class_names:
        items.append((name, CLASS_COLORS.get(name, COLOR_DEFAULT_CLS)))

    line_h, pad, box_w = 22, 8, 220
    box_h = pad * 2 + line_h * len(items)
    x0, y0 = w - box_w - 10, 10

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, dst=frame)
    cv2.rectangle(frame, (x0, y0), (x0 + box_w, y0 + box_h), (200, 200, 200), 1)

    for i, (name, color) in enumerate(items):
        y = y0 + pad + i * line_h + line_h // 2
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7), color, -1)
        cv2.rectangle(frame, (x0 + pad, y - 7), (x0 + pad + 20, y + 7),
                      (255, 255, 255), 1)
        cv2.putText(frame, name, (x0 + pad + 30, y + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_TEXT, 1, cv2.LINE_AA)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main():
    # Validate
    required = [(VIDEO_INPUT, "video"),
                (DET_MODEL_PATH, "YOLO-det"),
                (CLIP_CHECKPOINT, "CLIP checkpoint")]
    if USE_ROI:
        required.append((ROI_JSON, "ROI JSON"))
    for p, name in required:
        if not p.exists():
            print(f"[ERROR] Không tìm thấy {name}: {p}")
            return

    mode_str = (f"USE_ROI=True (threshold={ROI_COVERAGE_THRESHOLD:.0%})"
                if USE_ROI else "USE_ROI=False (xử lý cả frame)")
    print(f"Mode: {mode_str}")
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # Load YOLO-det
    print("\nLoad YOLO-det...")
    det_model = YOLO(str(DET_MODEL_PATH))
    print(f"  Det classes: {det_model.names}")

    # Load CLIP classifier
    print()
    cls_model = CLIPInference(CLIP_CHECKPOINT, device=DEVICE)

    # Mở video
    cap = cv2.VideoCapture(str(VIDEO_INPUT))
    if not cap.isOpened():
        print(f"[ERROR] Không mở được video: {VIDEO_INPUT}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"\nVideo: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames")

    # ROI
    roi_polygon, roi_mask = None, None
    if USE_ROI:
        roi_polygon = load_roi(ROI_JSON, (height, width))
        roi_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.fillPoly(roi_mask, [roi_polygon], 255)

    # Writer
    VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(VIDEO_OUTPUT),
                             cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (width, height))

    frame_idx = 0
    stats = {"total_det": 0, "in_roi": 0, "classified": 0}

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            # YOLO-det
            det_kwargs = dict(conf=DET_CONF_THRESHOLD, verbose=False, device=DEVICE)
            if VEHICLE_CLASS_IDS is not None:
                det_kwargs["classes"] = VEHICLE_CLASS_IDS
            det_results = det_model.predict(frame, **det_kwargs)[0]

            if USE_ROI:
                draw_roi(frame, roi_polygon)

            # Duyệt bbox
            if det_results.boxes is not None and len(det_results.boxes) > 0:
                boxes = det_results.boxes.xyxy.cpu().numpy()

                for bbox in boxes:
                    stats["total_det"] += 1
                    x1, y1, x2, y2 = map(int, bbox)
                    if x2 <= x1 or y2 <= y1:
                        continue

                    if USE_ROI:
                        coverage = bbox_roi_coverage((x1, y1, x2, y2), roi_mask)
                        if coverage < ROI_COVERAGE_THRESHOLD:
                            draw_bbox(frame, (x1, y1, x2, y2), COLOR_VEHICLE)
                            continue

                    stats["in_roi"] += 1

                    # Crop + CLIP classify
                    x1c, y1c = max(0, x1), max(0, y1)
                    x2c, y2c = min(width, x2), min(height, y2)
                    crop = frame[y1c:y2c, x1c:x2c]

                    cls_label, cls_conf, _ = cls_model.classify(crop)

                    if cls_label is not None and cls_conf >= CLS_CONF_THRESHOLD:
                        stats["classified"] += 1
                        label_text = f"{cls_label} {cls_conf:.2f}"
                        box_color = CLASS_COLORS.get(cls_label, COLOR_DEFAULT_CLS)
                    else:
                        label_text = "?"
                        box_color = COLOR_DEFAULT_CLS

                    draw_bbox(frame, (x1, y1, x2, y2), box_color, label=label_text)

            # Overlay
            info_label = "classified" if not USE_ROI else "in_roi"
            info = f"Frame {frame_idx}/{total_frames} | {info_label}: {stats['in_roi']}"
            cv2.putText(frame, info, (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            draw_legend(frame, cls_model.class_names, show_out_roi=USE_ROI)

            writer.write(frame)

            if SHOW_REALTIME:
                cv2.imshow("Pipeline (CLIP fine-tuned)", frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    print("\nDừng bởi người dùng.")
                    break

            if frame_idx % 30 == 0:
                pct = 100 * frame_idx / max(total_frames, 1)
                print(f"  Frame {frame_idx}/{total_frames} ({pct:.1f}%)  "
                      f"classified_sofar={stats['classified']}")

    finally:
        cap.release()
        writer.release()
        if SHOW_REALTIME:
            cv2.destroyAllWindows()

    # Báo cáo
    print("\n" + "=" * 60)
    print("HOÀN THÀNH")
    print("=" * 60)
    print(f"Frames xử lý        : {frame_idx}")
    print(f"Tổng detections     : {stats['total_det']}")
    if USE_ROI:
        print(f"  Pass ROI filter   : {stats['in_roi']}")
    print(f"  Đã classify       : {stats['classified']}")
    print(f"Video output        : {VIDEO_OUTPUT}")


if __name__ == "__main__":
    main()

/home/dmin/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mode: USE_ROI=False (xử lý cả frame)
Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti

Load YOLO-det...
  Det classes: {0: 'vehicle'}

Load CLIP checkpoint: clip_finetuned_1500/best_model.pt


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2060.49it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEX

  Model: openai/clip-vit-large-patch14
  Classes (9): ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']
  Input size: 224x224

Video: 1280x720 @ 30.0 FPS, 6023 frames
  Frame 30/6023 (0.5%)  classified_sofar=747
  Frame 60/6023 (1.0%)  classified_sofar=1530
  Frame 90/6023 (1.5%)  classified_sofar=2386
  Frame 120/6023 (2.0%)  classified_sofar=3234
  Frame 150/6023 (2.5%)  classified_sofar=4051
  Frame 180/6023 (3.0%)  classified_sofar=4855
  Frame 210/6023 (3.5%)  classified_sofar=5578
  Frame 240/6023 (4.0%)  classified_sofar=6172
  Frame 270/6023 (4.5%)  classified_sofar=6761
  Frame 300/6023 (5.0%)  classified_sofar=7372
  Frame 330/6023 (5.5%)  classified_sofar=7979
  Frame 360/6023 (6.0%)  classified_sofar=8592
  Frame 390/6023 (6.5%)  classified_sofar=9220
  Frame 420/6023 (7.0%)  classified_sofar=9888
  Frame 450/6023 (7.5%)  classified_sofar=10592
  Frame 480/6023 (8.0%)  classified_sofar=11306
  Frame 510/6023 (8.5%)  classified_sofar=12060

In [ ]:
"""
Train CLIP ViT-Large/14 (full fine-tune) for image classification.
"""

import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from transformers import CLIPImageProcessor, CLIPVisionModel


# ============================================================
# CẤU HÌNH
# ============================================================
DATA_DIR       = Path("/mnt/c/Users/Admin/Downloads/2026-ResearchWork/SmartM2M_AdmissionTest/Vehicle-Classification_Dataset-short-1500")   
OUTPUT_DIR     = Path("./clip_finetuned_1500_base")

MODEL_NAME     = "openai/clip-vit-base-patch32"

# Training
EPOCHS         = 5
BATCH_SIZE     = 16              # giảm nếu OOM
GRAD_ACCUM     = 2               # effective bs = BATCH_SIZE × GRAD_ACCUM
LR_HEAD        = 1e-3            # LR cho classifier head
LR_BACKBONE    = 1e-5            # LR cho CLIP vision encoder (nhỏ hơn nhiều)
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.1
LABEL_SMOOTH   = 0.1

# Memory optimization
USE_FP16       = True            # mixed precision
GRAD_CHECKPOINT = True           # tiết kiệm VRAM, chậm hơn ~20%

# Augmentation
IMG_SIZE       = 224             # CLIP-Large/14 dùng 224
AUG_STRENGTH   = "medium"        # "none" | "light" | "medium" | "strong"

# Misc
NUM_WORKERS    = 4
SEED           = 42
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_BEST_ONLY = True
EARLY_STOP_PATIENCE = 5
# ============================================================


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class ImageFolderDataset(Dataset):
    """Tự implement ImageFolder để có full control với CLIP processor."""

    IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def __init__(self, root, class_to_idx, transform):
        self.root = Path(root)
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.samples = []
        for cls_name, cls_idx in class_to_idx.items():
            cls_dir = self.root / cls_name
            if not cls_dir.is_dir():
                continue
            for p in cls_dir.iterdir():
                if p.suffix.lower() in self.IMG_EXT:
                    self.samples.append((p, cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img_tensor = self.transform(img)
        return img_tensor, label


def build_transforms(image_processor, strength="medium"):
    """
    Transform khớp với CLIP preprocessing + augmentation.
    CLIP chuẩn hóa bằng mean/std riêng, lấy từ processor.
    """
    mean = image_processor.image_mean
    std  = image_processor.image_std
    size = image_processor.size.get("shortest_edge", IMG_SIZE)

    normalize = transforms.Normalize(mean=mean, std=std)

    if strength == "none":
        train_aug = [transforms.Resize((size, size))]
    elif strength == "light":
        train_aug = [
            transforms.Resize((size + 32, size + 32)),
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(),
        ]
    elif strength == "medium":
        train_aug = [
            transforms.Resize((size + 32, size + 32)),
            transforms.RandomCrop(size),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.RandomRotation(10),
        ]
    else:  # strong
        train_aug = [
            transforms.Resize((size + 48, size + 48)),
            transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.1),
            transforms.RandomRotation(15),
            transforms.RandomErasing(p=0.25),
        ]

    train_tfm = transforms.Compose(train_aug + [transforms.ToTensor(), normalize])
    val_tfm   = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        normalize,
    ])
    return train_tfm, val_tfm


# ------------------------------------------------------------
# Model: CLIP vision encoder + linear classifier head
# ------------------------------------------------------------
class CLIPClassifier(nn.Module):
    """
    CLIP vision encoder + classifier head.
    Lấy [CLS] pooler output làm feature (dim=768 cho Base, 1024 cho Large).
    """
    def __init__(self, model_name, num_classes, grad_checkpoint=False):
        super().__init__()
        self.vision = CLIPVisionModel.from_pretrained(model_name)
        hidden = self.vision.config.hidden_size  # 1024 với Large/14
        if grad_checkpoint:
            self.vision.gradient_checkpointing_enable()
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, num_classes)

    def forward(self, pixel_values):
        out = self.vision(pixel_values=pixel_values)
        # pooler_output: [B, hidden]
        feats = out.pooler_output
        feats = self.dropout(feats)
        logits = self.head(feats)
        return logits


# ------------------------------------------------------------
# Train / Eval loop
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, scheduler, scaler,
                    criterion, epoch, total_epochs):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [train]")

    optimizer.zero_grad()
    for step, (imgs, labels) in enumerate(pbar):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_FP16):
            logits = model(imgs)
            loss = criterion(logits, labels) / GRAD_ACCUM

        if USE_FP16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Gradient accumulation
        if (step + 1) % GRAD_ACCUM == 0:
            if USE_FP16:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        # Thống kê
        total_loss += loss.item() * GRAD_ACCUM * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{total_loss / total:.4f}",
            "acc":  f"{correct / total:.4f}",
            "lr":   f"{scheduler.get_last_lr()[0]:.2e}",
        })

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, desc="val"):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for imgs, labels in tqdm(loader, desc=f"[{desc}]"):
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_FP16):
            logits = model(imgs)
            loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (total_loss / total, correct / total,
            np.array(all_preds), np.array(all_labels))


def main():
    set_seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Device: {DEVICE}")
    print(f"Model : {MODEL_NAME}")
    if DEVICE == "cuda":
        print(f"GPU   : {torch.cuda.get_device_name(0)}, "
              f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Validate
    for split in ("train", "val"):
        if not (DATA_DIR / split).is_dir():
            print(f"[ERROR] Không có folder {DATA_DIR / split}")
            return

    # Class mapping từ train folder
    train_dir = DATA_DIR / "train"
    class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    print(f"\nClasses ({num_classes}): {class_names}")

    # Load CLIP processor để lấy mean/std/size
    print("\nLoad CLIP image processor...")
    image_processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)
    train_tfm, val_tfm = build_transforms(image_processor, AUG_STRENGTH)

    # Dataset + loader
    train_set = ImageFolderDataset(train_dir, class_to_idx, train_tfm)
    val_set   = ImageFolderDataset(DATA_DIR / "val", class_to_idx, val_tfm)
    test_set  = None
    if (DATA_DIR / "test").is_dir():
        test_set = ImageFolderDataset(DATA_DIR / "test", class_to_idx, val_tfm)

    print(f"Train: {len(train_set)}, Val: {len(val_set)}"
          + (f", Test: {len(test_set)}" if test_set else ""))

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = None
    if test_set:
        test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

    # Model
    print(f"\nLoad model {MODEL_NAME}...")
    model = CLIPClassifier(MODEL_NAME, num_classes,
                           grad_checkpoint=GRAD_CHECKPOINT).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Params: {n_params/1e6:.1f}M total, {n_trainable/1e6:.1f}M trainable")

    # Optimizer với 2 LR khác nhau: backbone nhỏ, head lớn
    backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
    head_params     = [p for n, p in model.named_parameters() if "head" in n]
    optimizer = AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params,     "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    # Scheduler: warmup + cosine
    steps_per_epoch = len(train_loader) // GRAD_ACCUM
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Loss + scaler
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

    # Training loop
    print(f"\nBắt đầu train: epochs={EPOCHS}, bs={BATCH_SIZE}, "
          f"effective_bs={BATCH_SIZE * GRAD_ACCUM}, fp16={USE_FP16}")
    print(f"Total steps: {total_steps}, warmup: {warmup_steps}\n")

    best_val_acc = 0.0
    no_improve = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            criterion, epoch, EPOCHS
        )
        val_loss, val_acc, val_preds, val_labels = evaluate(
            model, val_loader, criterion, desc="val"
        )

        print(f"\nEpoch {epoch}: "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        history.append({
            "epoch": epoch,
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
        })

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve = 0
            ckpt_path = OUTPUT_DIR / "best_model.pt"
            torch.save({
                "model_state": model.state_dict(),
                "class_to_idx": class_to_idx,
                "class_names": class_names,
                "model_name": MODEL_NAME,
                "epoch": epoch,
                "val_acc": val_acc,
            }, ckpt_path)
            print(f"  [✓] Lưu best model (val_acc={val_acc:.4f}) -> {ckpt_path}")
        else:
            no_improve += 1
            if not SAVE_BEST_ONLY:
                last_path = OUTPUT_DIR / f"epoch_{epoch:02d}.pt"
                torch.save({"model_state": model.state_dict(),
                            "epoch": epoch, "val_acc": val_acc}, last_path)

        # Early stopping
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping sau {no_improve} epoch không cải thiện.")
            break

    # Lưu history
    with open(OUTPUT_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Test
    if test_loader is not None:
        print("\n" + "=" * 60)
        print("Đánh giá trên TEST (dùng best model)")
        print("=" * 60)
        ckpt = torch.load(OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        test_loss, test_acc, test_preds, test_labels = evaluate(
            model, test_loader, criterion, desc="test"
        )
        print(f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}")

        # Per-class accuracy
        print("\nAccuracy theo class:")
        for cls_name, cls_idx in class_to_idx.items():
            mask = test_labels == cls_idx
            if mask.sum() > 0:
                acc = (test_preds[mask] == cls_idx).mean()
                print(f"  {cls_name:15s}: {acc:.4f} ({mask.sum()} mẫu)")

        # Confusion matrix nếu có sklearn
        try:
            from sklearn.metrics import classification_report
            print("\nClassification report:")
            print(classification_report(test_labels, test_preds,
                                        target_names=class_names, digits=4))
        except ImportError:
            pass

    print(f"\n[✓] Hoàn thành. Best val_acc = {best_val_acc:.4f}")
    print(f"    Checkpoint tốt nhất: {OUTPUT_DIR / 'best_model.pt'}")


if __name__ == "__main__":
    main()

/home/dmin/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Model : openai/clip-vit-base-patch32
GPU   : NVIDIA GeForce RTX 4070 Ti, 12.9 GB

Classes (9): ['Bus', 'MPV', 'SUV', 'Sedan', 'Sports Car', 'Truck', 'Van', 'bicycle', 'motorcycle']

Load CLIP image processor...
Train: 9450, Val: 4050

Load model openai/clip-vit-base-patch32...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 75296.93it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEX

Params: 87.5M total, 87.5M trainable

Bắt đầu train: epochs=5, bs=16, effective_bs=32, fp16=True
Total steps: 1475, warmup: 147



Epoch 1/5 [train]:   0%|          | 0/590 [00:00<?, ?it/s]/tmp/ipykernel_71153/3705778215.py:202: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
[val]:   0%|          | 0/254 [00:00<?, ?it/s]/tmp/ipykernel_71153/3705778215.py:249: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
[val]: 100%|██████████| 254/254 [00:06<00:00, 38.32it/s]


...


Epoch 3: train_loss=0.6847 train_acc=0.9175 | val_loss=0.7271 val_acc=0.8928
  [✓] Lưu best model (val_acc=0.8928) -> clip_finetuned_1500_base/best_model.pt


[val]: 100%|██████████| 254/254 [00:06<00:00, 38.16it/s]



Epoch 4: train_loss=0.5973 train_acc=0.9560 | val_loss=0.7013 val_acc=0.9059
  [✓] Lưu best model (val_acc=0.9059) -> clip_finetuned_1500_base/best_model.pt


[val]: 100%|██████████| 254/254 [00:06<00:00, 37.43it/s]


Epoch 5: train_loss=0.5596 train_acc=0.9738 | val_loss=0.6998 val_acc=0.9042

[✓] Hoàn thành. Best val_acc = 0.9059
    Checkpoint tốt nhất: clip_finetuned_1500_base/best_model.pt
